# 02 - Entrenamiento de los modelos

Entrena la regresión logística (baseline) y el MLP programados en numpy (`src/models/`),
muestra las curvas de error y corre los experimentos de hiperparámetros. Todo el código vive
en `src/`; este notebook solo lo llama.

Funciona en **Google Colab** y en local (VS Code / Jupyter). En Colab hay que subir antes a
Google Drive la carpeta `lcc_64_gris` generada por `preparar_matrices` (train/val/test/normalizacion.npz)
y ajustar `DRIVE_MATRICES` abajo.

In [ ]:
import os
import sys
from pathlib import Path

EN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Alous12/Proyecto_IA.git"
DRIVE_MATRICES = "/content/drive/MyDrive/Proyecto_IA/matrices/lcc_64_gris"

if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    if not Path("/content/Proyecto_IA").exists():
        !git clone -q {REPO_URL} /content/Proyecto_IA
    RAIZ = Path("/content/Proyecto_IA")
    destino = RAIZ / "data/processed/matrices/lcc_64_gris"
    destino.mkdir(parents=True, exist_ok=True)
    !cp -n {DRIVE_MATRICES}/*.npz {destino}/
else:
    RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(RAIZ)
sys.path.insert(0, str(RAIZ))
print("Raíz del proyecto:", RAIZ)

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from src import config
from src.data.dataset import cargar_datos
from src.models.funciones_comunes import calcular_peso_real
from src.train import ejecutar

X_train, y_train, info_train = cargar_datos("train")
X_val, y_val, info_val = cargar_datos("val")
X_test, y_test, info_test = cargar_datos("test")
for nombre, X, y in [("train", X_train, y_train), ("val", X_val, y_val), ("test", X_test, y_test)]:
    print(f"{nombre:5s} X {X.shape}  ataques {y.mean():.1%}")
print(f"\nPeso de la clase real en la BCE (n_ataques / n_reales en train): {calcular_peso_real(y_train):.2f}")

## 1. Verificación del gradiente

Antes de entrenar se comprueba que backpropagation coincide con el gradiente numérico
(diferencias centrales) en un ejemplo pequeño. Error relativo < 1e-7 = derivadas correctas.

In [ ]:
from src import verificar_gradiente

verificar_gradiente.main()

## 2. Regresión logística (baseline)

$p = \sigma(XW + b)$, entrenada con Adam y la BCE ponderada para compensar el desbalance
(~85 % ataques). Se guardan los parámetros de la época con menor error de validación.

In [ ]:
def mostrar(resumen):
    carpeta = config.ruta_corrida(resumen["nombre"])
    display(Image(carpeta / "curva_error.png", width=600))
    display(Image(carpeta / "matriz_confusion.png", width=650))
    print(json.dumps({k: resumen[k] for k in ["mejor_epoca", "val_acer", "test_acer_frame",
                                               "test_auc_frame", "segundos"]}, indent=2))

resumen_rl = ejecutar({"modelo": "regresion_logistica", "optimizador": "adam", "lr": 1e-3,
                       "lote": 128, "epocas": 60, "paciencia": 15})
mostrar(resumen_rl)

## 3. MLP

Capas ocultas ReLU con dropout y salida sigmoide. Arquitectura base 4096 → 256 → 64 → 1.

In [ ]:
resumen_mlp = ejecutar({"modelo": "mlp", "capas": [256, 64], "dropout": 0.3, "optimizador": "adam",
                        "lr": 1e-3, "lote": 128, "epocas": 60, "paciencia": 15})
mostrar(resumen_mlp)

## 4. Experimentos de hiperparámetros

Se cambia un factor a la vez respecto a la configuración base: optimizador (GD, minibatch, Adam),
tasa de aprendizaje, tamaño de lote, arquitectura, dropout, L2 y balance de clases. Las corridas
ya hechas con los mismos datos (misma huella) no se repiten. Tarda unos minutos.

In [ ]:
!{sys.executable} -m src.experimentos

In [ ]:
tabla = pd.read_csv(config.RUTA_EXPERIMENTOS_CSV)
columnas = ["grupo", "nombre", "mejor_epoca", "error_val", "val_acer", "val_auc", "test_acer_frame", "segundos"]
tabla[columnas].sort_values(["grupo", "val_acer"]).style.format(precision=3).hide(axis="index")

In [ ]:
for ruta in sorted(config.DIR_FIGURAS_GRUPOS.glob("*.png")):
    print(ruta.stem)
    display(Image(ruta, width=800))

## 5. Modelos finales

El mejor modelo de cada tipo se elige por ACER en **validación**; el test solo se reporta.

In [ ]:
mejores = json.load(open(config.RUTA_MEJORES_MODELOS, encoding="utf-8"))
for modelo, fila in mejores.items():
    print(f"{modelo}: {fila['nombre']}")
    carpeta = config.ruta_final(modelo)
    display(Image(carpeta / "curva_error.png", width=600))
    display(Image(carpeta / "curva_roc.png", width=450))

finales = pd.DataFrame([{"modelo": m, **{k: f[k] for k in ["nombre", "val_acer", "test_apcer_frame",
                        "test_bpcer_frame", "test_acer_frame", "test_auc_frame"]}}
                        for m, f in mejores.items()])
finales.style.format(precision=3).hide(axis="index")